In [2]:
import pandas as pd
import numpy as np

In [3]:
df1=pd.read_csv(r"D:\code_of_digital_impersonation\unique_professional_identities_500_repaired.csv")
df2=pd.read_csv(r"D:\code_of_digital_impersonation\dataset2_500_final_v7.csv")
df3=pd.read_csv(r"D:\code_of_digital_impersonation\dataset3_500_final_v4.csv")

# final code 

In [17]:
# ============================================================
# CYBERGUARD — COMPLETE END-TO-END PIPELINE
# Dataset 1 → Model 1 → Model 2 → Model 3 → Risk Engine
# ============================================================

import joblib
import pandas as pd
import re

# ============================================================
# 1. LOAD SAVED MODELS
# ============================================================

model_1 = joblib.load(
    "cyberguard_model_1_impersonation.pkl"
)

tfidf_1 = joblib.load(
    "cyberguard_model_1_tfidf.pkl"
)

model_2 = joblib.load(
    "cyberguard_model_2_attack_type.pkl"
)

tfidf_2 = joblib.load(
    "cyberguard_model_2_tfidf.pkl"
)

model_3 = joblib.load(
    "cyberguard_model_3_behavioral.pkl"
)

# Dataset 1 must already be loaded as df1
if "df1" not in globals():
    raise RuntimeError(
        "Dataset 1 is not loaded as df1."
    )

# ============================================================
# 2. EMAIL INPUT
# ============================================================
# Change ONLY this section when testing another email.

sender_name = "Rajesh Kumar"
sender_email = "raj.kumar@gmail.com"

receiver_name = "Finance Officer"
receiver_email = "finance@example.org"

reply_to = ""

date_time = "2026-09-23 14:30"

subject = "Urgent: Verify your account immediately"

message = """
Dear Finance Officer,

Your account requires immediate verification. Please confirm your
login credentials and verification code as soon as possible to avoid
suspension of your account.

Regards,
Rajesh Kumar
"""

# ============================================================
# 3. BASIC EXTRACTION
# ============================================================

text_input = subject + " " + message

sender_domain = (
    sender_email.split("@")[-1].lower()
    if "@" in sender_email
    else ""
)

receiver_domain = (
    receiver_email.split("@")[-1].lower()
    if "@" in receiver_email
    else ""
)

reply_to_domain = (
    reply_to.split("@")[-1].lower()
    if reply_to and "@" in reply_to
    else ""
)

message_length = len(message.strip())

dt = pd.to_datetime(date_time)

current_hour = dt.hour
current_day = dt.day_name()

if 5 <= current_hour < 12:
    current_period = "Morning"
elif 12 <= current_hour < 17:
    current_period = "Afternoon"
elif 17 <= current_hour < 21:
    current_period = "Evening"
else:
    current_period = "Night"

# ============================================================
# 4. DATASET 1 — IDENTITY ANALYSIS
# ============================================================

identity_matches = df1[
    df1["official_email"]
    .astype(str)
    .str.lower()
    == sender_email.lower()
]

identity_found = len(identity_matches) > 0

if identity_found:

    identity = identity_matches.iloc[0]

    identity_id = identity["identity_id"]
    identity_name = identity["name"]
    identity_role = identity["role"]
    identity_organization = identity["organization"]
    official_domain = str(
        identity["official_domain"]
    ).lower()

else:

    identity = None

    identity_id = None
    identity_name = None
    identity_role = None
    identity_organization = None
    official_domain = None

# ============================================================
# 5. MODEL 1 — IMPERSONATION DETECTION
# ============================================================

X_m1 = tfidf_1.transform(
    [text_input]
)

m1_probabilities = model_1.predict_proba(
    X_m1
)[0]

legitimate_probability = float(
    m1_probabilities[0]
)

impersonation_probability = float(
    m1_probabilities[1]
)

if impersonation_probability >= 0.5:
    model1_prediction = "IMPERSONATION"
else:
    model1_prediction = "LEGITIMATE"

# ============================================================
# 6. MODEL 2 — ATTACK TYPE CLASSIFICATION
# ============================================================

X_m2 = tfidf_2.transform(
    [text_input]
)

m2_probabilities = model_2.predict_proba(
    X_m2
)[0]

m2_index = m2_probabilities.argmax()

model2_prediction = str(
    model_2.classes_[m2_index]
)

model2_confidence = float(
    m2_probabilities[m2_index]
)

# Top 3 attack types
top3_indices = m2_probabilities.argsort()[-3:][::-1]

top3_attacks = [
    (
        str(model_2.classes_[i]),
        float(m2_probabilities[i])
    )
    for i in top3_indices
]

# ============================================================
# 7. MODEL 3 — BEHAVIORAL ANALYSIS
# ============================================================

profiles = model_3["dataset3_profiles"]
weights = model_3["weights"]

model3_status = "UNKNOWN"
model3_score = None

behavior_details = {
    "time_match": None,
    "day_match": None,
    "domain_match": None,
    "length_match": None,
}

if identity_found:

    profile_matches = profiles[
        profiles["identity_id"]
        == identity_id
    ]

    if len(profile_matches) > 0:

        profile = profile_matches.iloc[0]

        # --------------------------------------------
        # Time
        # --------------------------------------------

        time_match = (
            current_period.lower()
            in str(profile["usual_hours"]).lower()
        )

        # --------------------------------------------
        # Day
        # --------------------------------------------

        usual_days = str(
            profile["usual_days"]
        ).lower()

        if "all days" in usual_days:

            day_match = True

        elif "weekdays" in usual_days:

            day_match = current_day not in [
                "Saturday",
                "Sunday"
            ]

        else:

            day_match = (
                current_day.lower()
                in usual_days
            )

        # --------------------------------------------
        # Sender domain
        # --------------------------------------------

        behavior_domain = str(
            profile["usual_sender_domain"]
        ).lower()

        domain_match = (
            sender_domain
            == behavior_domain
        )

        # --------------------------------------------
        # Message length
        # --------------------------------------------

        avg_length = float(
            profile["avg_message_length"]
        )

        if avg_length > 0:

            length_ratio = (
                message_length
                / avg_length
            )

            length_match = (
                0.5
                <= length_ratio
                <= 1.5
            )

        else:

            length_ratio = None
            length_match = False

        # --------------------------------------------
        # Weighted behavioral score
        # --------------------------------------------

        model3_score = (

            weights["time_of_day"]
            * (0 if time_match else 1)

            +

            weights["day_of_week"]
            * (0 if day_match else 1)

            +

            weights["sender_domain"]
            * (0 if domain_match else 1)

            +

            weights["message_length"]
            * (0 if length_match else 1)
        )

        if model3_score >= 0.5:
            model3_status = "ANOMALOUS"
        else:
            model3_status = "NORMAL"

        behavior_details = {
            "time_match": time_match,
            "day_match": day_match,
            "domain_match": domain_match,
            "length_match": length_match,
            "length_ratio": (
                length_ratio
                if avg_length > 0
                else None
            )
        }

# ============================================================
# 8. TECHNICAL / CONTENT SIGNALS
# ============================================================

public_email_domains = {
    "gmail.com",
    "yahoo.com",
    "outlook.com",
    "hotmail.com",
    "proton.me",
    "protonmail.com",
    "icloud.com"
}

public_email = (
    sender_domain
    in public_email_domains
)

# Sender domain vs Dataset 1
if identity_found:

    sender_domain_matches_identity = (
        sender_domain
        == official_domain
    )

else:

    sender_domain_matches_identity = False

# Reply-To
if reply_to:

    reply_to_matches_sender = (
        reply_to_domain
        == sender_domain
    )

else:

    # No Reply-To is not a mismatch
    reply_to_matches_sender = True

# ============================================================
# 9. CONTENT SIGNAL DETECTION
# ============================================================

lower_text = text_input.lower()

urgency_words = [
    "urgent",
    "immediately",
    "as soon as possible",
    "right away",
    "within",
    "suspended",
    "suspension",
    "today",
    "now"
]

sensitive_words = [
    "password",
    "login credentials",
    "credential",
    "verification code",
    "otp",
    "pin",
    "account details",
    "bank details",
    "financial information"
]

urgency_detected = any(
    word in lower_text
    for word in urgency_words
)

sensitive_request_detected = any(
    word in lower_text
    for word in sensitive_words
)

# ============================================================
# 10. LINK / ATTACHMENT SIGNALS
# ============================================================

url_pattern = r"https?://[^\s]+|www\.[^\s]+"

links = re.findall(
    url_pattern,
    text_input,
    flags=re.IGNORECASE
)

has_link = len(links) > 0

attachment_extensions = [
    ".pdf",
    ".doc",
    ".docx",
    ".xls",
    ".xlsx",
    ".zip",
    ".rar",
    ".exe"
]

has_attachment = any(
    ext in lower_text
    for ext in attachment_extensions
)

# ============================================================
# 11. RISK / EVIDENCE FUSION
# ============================================================

# Initial engineering weights.
# These are NOT calibrated probabilities.

MAX_MODEL1 = 25
UNKNOWN_IDENTITY = 10
DOMAIN_MISMATCH = 25
REPLY_TO_MISMATCH = 15
PUBLIC_EMAIL = 10
URGENCY = 15
SENSITIVE_REQUEST = 20
MAX_MODEL3 = 15

risk_contributions = {}

# Model 1
risk_contributions[
    "Model 1 impersonation"
] = (
    impersonation_probability
    * MAX_MODEL1
)

# Unknown identity
risk_contributions[
    "Unknown identity"
] = (
    UNKNOWN_IDENTITY
    if not identity_found
    else 0
)

# Domain mismatch
risk_contributions[
    "Domain mismatch"
] = (
    DOMAIN_MISMATCH
    if (
        identity_found
        and not sender_domain_matches_identity
    )
    else 0
)

# Reply-To mismatch
risk_contributions[
    "Reply-To mismatch"
] = (
    REPLY_TO_MISMATCH
    if not reply_to_matches_sender
    else 0
)

# Public email
risk_contributions[
    "Public email domain"
] = (
    PUBLIC_EMAIL
    if public_email
    else 0
)

# Urgency
risk_contributions[
    "Urgency"
] = (
    URGENCY
    if urgency_detected
    else 0
)

# Sensitive request
risk_contributions[
    "Sensitive request"
] = (
    SENSITIVE_REQUEST
    if sensitive_request_detected
    else 0
)

# Model 3
if model3_score is not None:

    risk_contributions[
        "Model 3 behavior"
    ] = (
        model3_score
        * MAX_MODEL3
    )

else:

    risk_contributions[
        "Model 3 behavior"
    ] = 0

# ============================================================
# 12. FINAL RISK SCORE
# ============================================================

risk_score = sum(
    risk_contributions.values()
)

risk_score = min(
    100,
    risk_score
)

# ============================================================
# 13. RISK BAND
# ============================================================

if risk_score <= 20:

    risk_band = "SAFE"

elif risk_score <= 40:

    risk_band = "LOW"

elif risk_score <= 60:

    risk_band = "MEDIUM"

elif risk_score <= 80:

    risk_band = "HIGH"

else:

    risk_band = "CRITICAL"

# ============================================================
# 14. EXPLANATION GENERATION
# ============================================================

reasons = []

if impersonation_probability >= 0.5:

    reasons.append(
        "Model 1 detected impersonation indicators."
    )

if not identity_found:

    reasons.append(
        "Sender identity was not found in Dataset 1."
    )

if public_email:

    reasons.append(
        "Sender uses a public/free email domain."
    )

if urgency_detected:

    reasons.append(
        "Message contains urgency indicators."
    )

if sensitive_request_detected:

    reasons.append(
        "Message requests sensitive authentication information."
    )

if (
    identity_found
    and not sender_domain_matches_identity
):

    reasons.append(
        "Sender domain does not match the known identity."
    )

if not reply_to_matches_sender:

    reasons.append(
        "Reply-To domain does not match the sender domain."
    )

if model3_status == "ANOMALOUS":

    reasons.append(
        "Sender behavior is anomalous for the identified identity."
    )

elif model3_status == "NORMAL":

    reasons.append(
        "Sender behavior is consistent with the learned profile."
    )

else:

    reasons.append(
        "Behavioral analysis is unavailable because "
        "the sender identity is unknown."
    )

# ============================================================
# 15. FINAL CYBERGUARD REPORT
# ============================================================

print("\n")
print("=" * 90)
print("                 CYBERGUARD — FINAL ANALYSIS")
print("=" * 90)

print("\n[1] EMAIL")
print("-" * 90)
print("Sender           :", sender_name)
print("Sender email     :", sender_email)
print("Receiver         :", receiver_name)
print("Receiver email   :", receiver_email)
print("Subject          :", subject)
print("Date/time        :", date_time)

print("\n[2] IDENTITY ANALYSIS — DATASET 1")
print("-" * 90)
print("Identity found   :", identity_found)

if identity_found:

    print("Identity ID      :", identity_id)
    print("Identity name    :", identity_name)
    print("Role             :", identity_role)
    print("Organization     :", identity_organization)
    print("Official domain  :", official_domain)

else:

    print("Identity ID      : None")
    print("Status           : UNKNOWN SENDER")

print("\n[3] MODEL 1 — IMPERSONATION DETECTION")
print("-" * 90)
print("Prediction       :", model1_prediction)
print(
    "Legitimate       :",
    f"{legitimate_probability:.4f}"
)
print(
    "Impersonation    :",
    f"{impersonation_probability:.4f}"
)

print("\n[4] MODEL 2 — ATTACK TYPE")
print("-" * 90)
print("Attack type      :", model2_prediction)
print(
    "Confidence       :",
    f"{model2_confidence:.4f}"
)

print("\nTop 3 attack types:")

for attack, probability in top3_attacks:

    print(
        f"  {attack:<25} "
        f"{probability:.4f}"
    )

print("\n[5] MODEL 3 — BEHAVIORAL ANALYSIS")
print("-" * 90)
print("Status           :", model3_status)

if model3_score is not None:

    print(
        "Behavior score   :",
        f"{model3_score:.4f}"
    )

    print(
        "Time match       :",
        behavior_details["time_match"]
    )

    print(
        "Day match        :",
        behavior_details["day_match"]
    )

    print(
        "Domain match     :",
        behavior_details["domain_match"]
    )

    print(
        "Length match     :",
        behavior_details["length_match"]
    )

else:

    print("Behavior score   : N/A")

print("\n[6] TECHNICAL / CONTENT ANALYSIS")
print("-" * 90)
print("Sender domain    :", sender_domain)
print("Receiver domain  :", receiver_domain)
print(
    "Public email     :",
    public_email
)
print(
    "Domain mismatch  :",
    (
        identity_found
        and not sender_domain_matches_identity
    )
)
print(
    "Reply-To mismatch:",
    not reply_to_matches_sender
)
print(
    "Urgency          :",
    urgency_detected
)
print(
    "Sensitive request:",
    sensitive_request_detected
)
print(
    "Has link         :",
    has_link
)
print(
    "Has attachment   :",
    has_attachment
)

print("\n[7] RISK / EVIDENCE FUSION")
print("-" * 90)

for name, value in risk_contributions.items():

    print(
        f"{name:<30}: +{value:.2f}"
    )

print("-" * 90)

print(
    "FINAL RISK SCORE :",
    f"{risk_score:.2f} / 100"
)

print(
    "RISK LEVEL       :",
    risk_band
)

print("\n[8] EXPLANATION")
print("-" * 90)

for i, reason in enumerate(
    reasons,
    start=1
):

    print(
        f"{i}. {reason}"
    )

print("\n" + "=" * 90)
print("                 CYBERGUARD ANALYSIS COMPLETE")
print("=" * 90)



                 CYBERGUARD — FINAL ANALYSIS

[1] EMAIL
------------------------------------------------------------------------------------------
Sender           : Rajesh Kumar
Sender email     : raj.kumar@gmail.com
Receiver         : Finance Officer
Receiver email   : finance@example.org
Subject          : Urgent: Verify your account immediately
Date/time        : 2026-09-23 14:30

[2] IDENTITY ANALYSIS — DATASET 1
------------------------------------------------------------------------------------------
Identity found   : False
Identity ID      : None
Status           : UNKNOWN SENDER

[3] MODEL 1 — IMPERSONATION DETECTION
------------------------------------------------------------------------------------------
Prediction       : IMPERSONATION
Legitimate       : 0.3049
Impersonation    : 0.6951

[4] MODEL 2 — ATTACK TYPE
------------------------------------------------------------------------------------------
Attack type      : CREDENTIAL_REQUEST
Confidence       : 0.4465

Top 

In [19]:
# ============================================================
# CYBERGUARD — COMPLETE END-TO-END PIPELINE
# ============================================================
#
# Dataset 1 → Identity Analysis
#          ↓
# Model 1 → Impersonation Detection
#          ↓
# Model 2 → Attack Type Classification
#          ↓
# Model 3 → Identity-Specific Behavioral Analysis
#          ↓
# Technical / Content Analysis
#          ↓
# Risk & Evidence Fusion
#          ↓
# Final Risk Score + Explanation
#
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

import joblib
import pandas as pd
import re


# ============================================================
# 1. LOAD SAVED MODELS
# ============================================================

model_1 = joblib.load(
    "cyberguard_model_1_impersonation.pkl"
)

tfidf_1 = joblib.load(
    "cyberguard_model_1_tfidf.pkl"
)

model_2 = joblib.load(
    "cyberguard_model_2_attack_type.pkl"
)

tfidf_2 = joblib.load(
    "cyberguard_model_2_tfidf.pkl"
)

model_3 = joblib.load(
    "cyberguard_model_3_behavioral.pkl"
)


# ============================================================
# 2. CHECK DATASET 1
# ============================================================

if "df1" not in globals():

    raise RuntimeError(
        "Dataset 1 is not loaded as df1."
    )


required_df1_columns = [
    "identity_id",
    "name",
    "role",
    "organization",
    "official_email",
    "official_domain"
]

missing_df1_columns = [
    col
    for col in required_df1_columns
    if col not in df1.columns
]

if missing_df1_columns:

    raise RuntimeError(
        f"Dataset 1 is missing columns: "
        f"{missing_df1_columns}"
    )


# ============================================================
# 3. EMAIL INPUT
# ============================================================
#
# Change ONLY this section when testing another email.
#
# ============================================================

sender_name = "Rajesh Kumar"

sender_email = "raj.kumar@gmail.com"

receiver_name = "Finance Officer"

receiver_email = "finance@example.org"

reply_to = ""

date_time = "2026-09-23 14:30"

subject = "Urgent: Verify your account immediately"

message = """
Dear Finance Officer,

Your account requires immediate verification. Please confirm your
login credentials and verification code as soon as possible to avoid
suspension of your account.

Regards,
Rajesh Kumar
"""


# ============================================================
# 4. BASIC INPUT VALIDATION
# ============================================================

sender_email = str(sender_email).strip()

receiver_email = str(receiver_email).strip()

reply_to = str(reply_to).strip()

sender_name = str(sender_name).strip()

receiver_name = str(receiver_name).strip()

subject = str(subject).strip()

message = str(message).strip()


if not sender_email:

    raise ValueError(
        "Sender email cannot be empty."
    )


if not subject:

    raise ValueError(
        "Subject cannot be empty."
    )


if not message:

    raise ValueError(
        "Message cannot be empty."
    )


# ============================================================
# 5. BASIC EXTRACTION
# ============================================================

text_input = (
    subject
    + " "
    + message
)


# Sender domain

sender_domain = (
    sender_email.split("@")[-1].lower()
    if "@" in sender_email
    else ""
)


# Receiver domain

receiver_domain = (
    receiver_email.split("@")[-1].lower()
    if "@" in receiver_email
    else ""
)


# Reply-To domain

reply_to_domain = (
    reply_to.split("@")[-1].lower()
    if reply_to and "@" in reply_to
    else ""
)


# Message length

message_length = len(
    message.strip()
)


# Date / time

dt = pd.to_datetime(
    date_time
)


current_hour = dt.hour

current_day = dt.day_name()


# Time period

if 5 <= current_hour < 12:

    current_period = "Morning"

elif 12 <= current_hour < 17:

    current_period = "Afternoon"

elif 17 <= current_hour < 21:

    current_period = "Evening"

else:

    current_period = "Night"


# ============================================================
# 6. DATASET 1 — ROBUST IDENTITY ANALYSIS
# ============================================================

def normalize_name(name):

    return re.sub(
        r"[^a-z0-9]",
        "",
        str(name).lower()
    )


# Find every identity using this official email

identity_matches = df1[
    df1["official_email"]
    .astype(str)
    .str.strip()
    .str.lower()
    ==
    sender_email.lower()
].copy()


# Default identity values

identity_found = False

identity_ambiguous = False

identity = None

identity_id = None

identity_name = None

identity_role = None

identity_organization = None

official_domain = None


# ------------------------------------------------------------
# CASE 1 — No identity found
# ------------------------------------------------------------

if len(identity_matches) == 0:

    identity_found = False

    identity_ambiguous = False


# ------------------------------------------------------------
# CASE 2 — Exactly one identity found
# ------------------------------------------------------------

elif len(identity_matches) == 1:

    identity_found = True

    identity_ambiguous = False

    identity = identity_matches.iloc[0]

    identity_id = identity["identity_id"]

    identity_name = identity["name"]

    identity_role = identity["role"]

    identity_organization = identity["organization"]

    official_domain = str(
        identity["official_domain"]
    ).strip().lower()


# ------------------------------------------------------------
# CASE 3 — Shared email
# ------------------------------------------------------------

else:

    normalized_sender_name = normalize_name(
        sender_name
    )

    identity_matches["normalized_name"] = (
        identity_matches["name"]
        .apply(normalize_name)
    )

    name_matches = identity_matches[
        identity_matches["normalized_name"]
        ==
        normalized_sender_name
    ].copy()


    # ----------------------------------------
    # Unique email + name match
    # ----------------------------------------

    if len(name_matches) == 1:

        identity_found = True

        identity_ambiguous = False

        identity = name_matches.iloc[0]

        identity_id = identity["identity_id"]

        identity_name = identity["name"]

        identity_role = identity["role"]

        identity_organization = identity["organization"]

        official_domain = str(
            identity["official_domain"]
        ).strip().lower()


    # ----------------------------------------
    # Still ambiguous
    # ----------------------------------------

    else:

        identity_found = False

        identity_ambiguous = True

        identity = None

        identity_id = None

        identity_name = None

        identity_role = None

        identity_organization = None

        official_domain = None


# ============================================================
# 7. MODEL 1 — IMPERSONATION DETECTION
# ============================================================

X_m1 = tfidf_1.transform(
    [text_input]
)


m1_probabilities = model_1.predict_proba(
    X_m1
)[0]


# Model 1 classes:
# 0 = LEGITIMATE
# 1 = IMPERSONATION

class_to_probability_m1 = {
    int(cls): float(prob)
    for cls, prob
    in zip(
        model_1.classes_,
        m1_probabilities
    )
}


legitimate_probability = (
    class_to_probability_m1.get(
        0,
        0.0
    )
)


impersonation_probability = (
    class_to_probability_m1.get(
        1,
        0.0
    )
)


if impersonation_probability >= 0.5:

    model1_prediction = "IMPERSONATION"

else:

    model1_prediction = "LEGITIMATE"


# ============================================================
# 8. MODEL 2 — ATTACK TYPE CLASSIFICATION
# ============================================================
#
# IMPORTANT:
# Model 2 was trained ONLY on IMPERSONATION records.
#
# Therefore Model 2 does NOT determine whether an email is
# legitimate or malicious.
#
# It identifies the likely attack type.
#
# ============================================================

X_m2 = tfidf_2.transform(
    [text_input]
)


m2_probabilities = model_2.predict_proba(
    X_m2
)[0]


m2_index = m2_probabilities.argmax()


model2_prediction = str(
    model_2.classes_[m2_index]
)


model2_confidence = float(
    m2_probabilities[m2_index]
)


# Top 3 attack types

top3_indices = (
    m2_probabilities
    .argsort()[-3:][::-1]
)


top3_attacks = [

    (
        str(model_2.classes_[i]),
        float(m2_probabilities[i])
    )

    for i in top3_indices
]


# ============================================================
# 9. MODEL 3 — IDENTITY-SPECIFIC BEHAVIORAL ANALYSIS
# ============================================================

profiles = model_3[
    "dataset3_profiles"
]

weights = model_3[
    "weights"
]


model3_status = "UNKNOWN"

model3_score = None


behavior_details = {

    "time_match": None,

    "day_match": None,

    "domain_match": None,

    "length_match": None,

    "length_ratio": None

}


# ------------------------------------------------------------
# Behavioral analysis only possible when identity is known
# ------------------------------------------------------------

if identity_found:

    profile_matches = profiles[
        profiles["identity_id"]
        ==
        identity_id
    ]


    if len(profile_matches) > 0:

        profile = profile_matches.iloc[0]


        # ----------------------------------------------------
        # TIME
        # ----------------------------------------------------

        time_match = (

            current_period.lower()

            in

            str(
                profile["usual_hours"]
            ).lower()

        )


        # ----------------------------------------------------
        # DAY
        # ----------------------------------------------------

        usual_days = str(
            profile["usual_days"]
        ).lower()


        if "all days" in usual_days:

            day_match = True


        elif "weekdays" in usual_days:

            day_match = (
                current_day
                not in
                [
                    "Saturday",
                    "Sunday"
                ]
            )


        else:

            day_match = (
                current_day.lower()
                in
                usual_days
            )


        # ----------------------------------------------------
        # SENDER DOMAIN
        # ----------------------------------------------------

        behavior_domain = str(
            profile["usual_sender_domain"]
        ).strip().lower()


        domain_match = (
            sender_domain
            ==
            behavior_domain
        )


        # ----------------------------------------------------
        # MESSAGE LENGTH
        # ----------------------------------------------------

        avg_length = float(
            profile["avg_message_length"]
        )


        if avg_length > 0:

            length_ratio = (
                message_length
                /
                avg_length
            )


            length_match = (
                0.5
                <=
                length_ratio
                <=
                1.5
            )


        else:

            length_ratio = None

            length_match = False


        # ----------------------------------------------------
        # BEHAVIORAL SCORE
        # ----------------------------------------------------
        #
        # 0 = completely consistent
        # 1 = all four behavioral signals anomalous
        #
        # ----------------------------------------------------

        model3_score = (

            weights["time_of_day"]
            *
            (0 if time_match else 1)

            +

            weights["day_of_week"]
            *
            (0 if day_match else 1)

            +

            weights["sender_domain"]
            *
            (0 if domain_match else 1)

            +

            weights["message_length"]
            *
            (0 if length_match else 1)

        )


        if model3_score >= 0.5:

            model3_status = "ANOMALOUS"

        else:

            model3_status = "NORMAL"


        behavior_details = {

            "time_match":
                time_match,

            "day_match":
                day_match,

            "domain_match":
                domain_match,

            "length_match":
                length_match,

            "length_ratio":
                length_ratio

        }


# ============================================================
# 10. TECHNICAL SIGNALS
# ============================================================

public_email_domains = {

    "gmail.com",

    "yahoo.com",

    "outlook.com",

    "hotmail.com",

    "proton.me",

    "protonmail.com",

    "icloud.com"

}


public_email = (
    sender_domain
    in public_email_domains
)


# ------------------------------------------------------------
# Sender domain vs known identity
# ------------------------------------------------------------

if identity_found:

    sender_domain_matches_identity = (

        sender_domain
        ==
        official_domain

    )

else:

    sender_domain_matches_identity = None


# ------------------------------------------------------------
# Reply-To
# ------------------------------------------------------------

if reply_to:

    reply_to_matches_sender = (

        reply_to_domain
        ==
        sender_domain

    )

else:

    # No Reply-To is NOT a mismatch.

    reply_to_matches_sender = True


# ============================================================
# 11. CONTENT SIGNAL DETECTION
# ============================================================

lower_text = text_input.lower()


# ------------------------------------------------------------
# Urgency
# ------------------------------------------------------------

urgency_words = [

    "urgent",

    "immediately",

    "as soon as possible",

    "right away",

    "within",

    "suspended",

    "suspension",

    "today",

    "now"

]


urgency_detected = any(

    word in lower_text

    for word in urgency_words

)


# ------------------------------------------------------------
# Sensitive information request
# ------------------------------------------------------------

sensitive_words = [

    "password",

    "login credentials",

    "credential",

    "verification code",

    "otp",

    "pin",

    "account details",

    "bank details",

    "financial information"

]


sensitive_request_detected = any(

    word in lower_text

    for word in sensitive_words

)


# ============================================================
# 12. LINK DETECTION
# ============================================================

url_pattern = (
    r"https?://[^\s]+"
    r"|www\.[^\s]+"
)


links = re.findall(

    url_pattern,

    text_input,

    flags=re.IGNORECASE

)


has_link = (
    len(links) > 0
)


# ============================================================
# 13. ATTACHMENT DETECTION
# ============================================================

attachment_extensions = [

    ".pdf",

    ".doc",

    ".docx",

    ".xls",

    ".xlsx",

    ".zip",

    ".rar",

    ".exe"

]


has_attachment = any(

    ext in lower_text

    for ext in attachment_extensions

)


# ============================================================
# 14. RISK / EVIDENCE FUSION
# ============================================================
#
# IMPORTANT:
# This is an engineering risk score.
# It is NOT a calibrated probability.
#
# ============================================================

MAX_MODEL1 = 25

UNKNOWN_IDENTITY = 10

DOMAIN_MISMATCH = 25

REPLY_TO_MISMATCH = 15

PUBLIC_EMAIL = 10

URGENCY = 15

SENSITIVE_REQUEST = 20

MAX_MODEL3 = 15


risk_contributions = {}


# ------------------------------------------------------------
# Model 1
# ------------------------------------------------------------

risk_contributions[
    "Model 1 impersonation"
] = (

    impersonation_probability
    *
    MAX_MODEL1

)


# ------------------------------------------------------------
# Unknown / ambiguous identity
# ------------------------------------------------------------
#
# Both UNKNOWN and AMBIGUOUS mean we cannot confidently
# establish the sender identity.
#
# ------------------------------------------------------------

risk_contributions[
    "Unknown / ambiguous identity"
] = (

    UNKNOWN_IDENTITY

    if (
        not identity_found
    )

    else 0

)


# ------------------------------------------------------------
# Domain mismatch
# ------------------------------------------------------------
#
# Only evaluate this when identity is actually known.
#
# Unknown sender ≠ domain mismatch.
#
# ------------------------------------------------------------

risk_contributions[
    "Domain mismatch"
] = (

    DOMAIN_MISMATCH

    if (

        identity_found

        and

        not sender_domain_matches_identity

    )

    else 0

)


# ------------------------------------------------------------
# Reply-To mismatch
# ------------------------------------------------------------

risk_contributions[
    "Reply-To mismatch"
] = (

    REPLY_TO_MISMATCH

    if not reply_to_matches_sender

    else 0

)


# ------------------------------------------------------------
# Public email domain
# ------------------------------------------------------------

risk_contributions[
    "Public email domain"
] = (

    PUBLIC_EMAIL

    if public_email

    else 0

)


# ------------------------------------------------------------
# Urgency
# ------------------------------------------------------------

risk_contributions[
    "Urgency"
] = (

    URGENCY

    if urgency_detected

    else 0

)


# ------------------------------------------------------------
# Sensitive request
# ------------------------------------------------------------

risk_contributions[
    "Sensitive request"
] = (

    SENSITIVE_REQUEST

    if sensitive_request_detected

    else 0

)


# ------------------------------------------------------------
# Model 3
# ------------------------------------------------------------

if model3_score is not None:

    risk_contributions[
        "Model 3 behavior"
    ] = (

        model3_score
        *
        MAX_MODEL3

    )

else:

    risk_contributions[
        "Model 3 behavior"
    ] = 0


# ============================================================
# 15. FINAL RISK SCORE
# ============================================================

risk_score = sum(
    risk_contributions.values()
)


risk_score = min(
    100,
    risk_score
)


# ============================================================
# 16. RISK BAND
# ============================================================

if risk_score <= 20:

    risk_band = "SAFE"

elif risk_score <= 40:

    risk_band = "LOW"

elif risk_score <= 60:

    risk_band = "MEDIUM"

elif risk_score <= 80:

    risk_band = "HIGH"

else:

    risk_band = "CRITICAL"


# ============================================================
# 17. EXPLANATION GENERATION
# ============================================================

reasons = []


# Model 1

if impersonation_probability >= 0.5:

    reasons.append(
        "Model 1 detected impersonation indicators."
    )


# Identity

if identity_ambiguous:

    reasons.append(
        "Sender email is shared by multiple Dataset 1 "
        "identities and could not be uniquely resolved."
    )

elif not identity_found:

    reasons.append(
        "Sender identity was not found in Dataset 1."
    )


# Public email

if public_email:

    reasons.append(
        "Sender uses a public/free email domain."
    )


# Domain mismatch

if (

    identity_found

    and

    not sender_domain_matches_identity

):

    reasons.append(
        "Sender domain does not match the known identity."
    )


# Reply-To

if not reply_to_matches_sender:

    reasons.append(
        "Reply-To domain does not match the sender domain."
    )


# Urgency

if urgency_detected:

    reasons.append(
        "Message contains urgency indicators."
    )


# Sensitive request

if sensitive_request_detected:

    reasons.append(
        "Message requests sensitive authentication "
        "or account information."
    )


# Link

if has_link:

    reasons.append(
        f"Message contains {len(links)} link(s)."
    )


# Attachment

if has_attachment:

    reasons.append(
        "Message appears to reference an attachment."
    )


# Model 3

if model3_status == "ANOMALOUS":

    reasons.append(
        "Sender behavior is anomalous for the "
        "identified identity."
    )

elif model3_status == "NORMAL":

    reasons.append(
        "Sender behavior is consistent with the "
        "learned identity profile."
    )

else:

    reasons.append(
        "Behavioral analysis is unavailable because "
        "the sender identity is unknown or ambiguous."
    )


# ============================================================
# 18. FINAL CYBERGUARD REPORT
# ============================================================

print("\n")

print("=" * 90)

print(
    "                 CYBERGUARD — FINAL ANALYSIS"
)

print("=" * 90)


# ------------------------------------------------------------
# EMAIL
# ------------------------------------------------------------

print("\n[1] EMAIL")

print("-" * 90)

print(
    "Sender           :",
    sender_name
)

print(
    "Sender email     :",
    sender_email
)

print(
    "Receiver         :",
    receiver_name
)

print(
    "Receiver email   :",
    receiver_email
)

print(
    "Subject          :",
    subject
)

print(
    "Date/time        :",
    date_time
)


# ------------------------------------------------------------
# DATASET 1
# ------------------------------------------------------------

print(
    "\n[2] IDENTITY ANALYSIS — DATASET 1"
)

print("-" * 90)

print(
    "Identity found   :",
    identity_found
)

print(
    "Identity ambiguous:",
    identity_ambiguous
)


if identity_found:

    print(
        "Identity ID      :",
        identity_id
    )

    print(
        "Identity name    :",
        identity_name
    )

    print(
        "Role             :",
        identity_role
    )

    print(
        "Organization     :",
        identity_organization
    )

    print(
        "Official domain  :",
        official_domain
    )


elif identity_ambiguous:

    print(
        "Status           : AMBIGUOUS SHARED EMAIL"
    )

    print(
        "Possible matches :",
        len(identity_matches)
    )


else:

    print(
        "Identity ID      : None"
    )

    print(
        "Status           : UNKNOWN SENDER"
    )


# ------------------------------------------------------------
# MODEL 1
# ------------------------------------------------------------

print(
    "\n[3] MODEL 1 — IMPERSONATION DETECTION"
)

print("-" * 90)

print(
    "Prediction       :",
    model1_prediction
)

print(
    "Legitimate       :",
    f"{legitimate_probability:.4f}"
)

print(
    "Impersonation    :",
    f"{impersonation_probability:.4f}"
)


# ------------------------------------------------------------
# MODEL 2
# ------------------------------------------------------------

print(
    "\n[4] MODEL 2 — ATTACK TYPE CLASSIFICATION"
)

print("-" * 90)

print(
    "Attack type      :",
    model2_prediction
)

print(
    "Confidence       :",
    f"{model2_confidence:.4f}"
)


print(
    "\nTop 3 attack types:"
)


for attack, probability in top3_attacks:

    print(
        f"  {attack:<25} "
        f"{probability:.4f}"
    )


# ------------------------------------------------------------
# MODEL 3
# ------------------------------------------------------------

print(
    "\n[5] MODEL 3 — BEHAVIORAL ANALYSIS"
)

print("-" * 90)

print(
    "Status           :",
    model3_status
)


if model3_score is not None:

    print(
        "Behavior score   :",
        f"{model3_score:.4f}"
    )

    print(
        "Time match       :",
        behavior_details["time_match"]
    )

    print(
        "Day match        :",
        behavior_details["day_match"]
    )

    print(
        "Domain match     :",
        behavior_details["domain_match"]
    )

    print(
        "Length match     :",
        behavior_details["length_match"]
    )

    print(
        "Length ratio     :",
        (
            f"{behavior_details['length_ratio']:.4f}"
            if behavior_details["length_ratio"]
            is not None
            else "N/A"
        )
    )

else:

    print(
        "Behavior score   : N/A"
    )


# ------------------------------------------------------------
# TECHNICAL / CONTENT
# ------------------------------------------------------------

print(
    "\n[6] TECHNICAL / CONTENT ANALYSIS"
)

print("-" * 90)

print(
    "Sender domain    :",
    sender_domain
)

print(
    "Receiver domain  :",
    receiver_domain
)

print(
    "Public email     :",
    public_email
)

print(
    "Domain mismatch  :",
    (
        identity_found
        and
        not sender_domain_matches_identity
    )
)

print(
    "Reply-To mismatch:",
    not reply_to_matches_sender
)

print(
    "Urgency          :",
    urgency_detected
)

print(
    "Sensitive request:",
    sensitive_request_detected
)

print(
    "Has link         :",
    has_link
)

print(
    "Link count       :",
    len(links)
)

print(
    "Has attachment   :",
    has_attachment
)


# ------------------------------------------------------------
# RISK ENGINE
# ------------------------------------------------------------

print(
    "\n[7] RISK / EVIDENCE FUSION"
)

print("-" * 90)


for name, value in risk_contributions.items():

    print(
        f"{name:<32}: +{value:.2f}"
    )


print("-" * 90)

print(
    "FINAL RISK SCORE :",
    f"{risk_score:.2f} / 100"
)

print(
    "RISK LEVEL       :",
    risk_band
)

print(
    "NOTE             : Engineering risk score, "
    "not a calibrated probability."
)


# ------------------------------------------------------------
# EXPLANATION
# ------------------------------------------------------------

print(
    "\n[8] EXPLANATION"
)

print("-" * 90)


for i, reason in enumerate(
    reasons,
    start=1
):

    print(
        f"{i}. {reason}"
    )


print("\n" + "=" * 90)

print(
    "                 CYBERGUARD ANALYSIS COMPLETE"
)

print("=" * 90)



                 CYBERGUARD — FINAL ANALYSIS

[1] EMAIL
------------------------------------------------------------------------------------------
Sender           : Rajesh Kumar
Sender email     : raj.kumar@gmail.com
Receiver         : Finance Officer
Receiver email   : finance@example.org
Subject          : Urgent: Verify your account immediately
Date/time        : 2026-09-23 14:30

[2] IDENTITY ANALYSIS — DATASET 1
------------------------------------------------------------------------------------------
Identity found   : False
Identity ambiguous: False
Identity ID      : None
Status           : UNKNOWN SENDER

[3] MODEL 1 — IMPERSONATION DETECTION
------------------------------------------------------------------------------------------
Prediction       : IMPERSONATION
Legitimate       : 0.3049
Impersonation    : 0.6951

[4] MODEL 2 — ATTACK TYPE CLASSIFICATION
------------------------------------------------------------------------------------------
Attack type      : CREDENTIA

In [21]:

import sys
import subprocess
print(sys.executable)

d:\abhijkk\my_env\Scripts\python.exe


In [22]:
import sys
import os

print("Jupyter Kernel:")
print("mlkernel (Python 3.13.2)")

print("\nPython executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

print("\nEnvironment prefix:")
print(sys.prefix)

print("\nBase prefix:")
print(sys.base_prefix)

print("\nIs this a virtual environment?")
print(sys.prefix != sys.base_prefix)

print("\nEnvironment folder exists?")
print(os.path.isdir(sys.prefix))

print("\nExpected environment:")
print(r"D:\abhijitkk\my_env")

print("\nCorrect environment?")
print(
    os.path.normcase(sys.prefix)
    == os.path.normcase(r"D:\abhijitkk\my_env")
)

Jupyter Kernel:
mlkernel (Python 3.13.2)

Python executable:
d:\abhijkk\my_env\Scripts\python.exe

Python version:
3.13.2 (tags/v3.13.2:4f8bb39, Feb  4 2025, 15:23:48) [MSC v.1942 64 bit (AMD64)]

Environment prefix:
d:\abhijkk\my_env

Base prefix:
C:\Users\ASUS\AppData\Local\Programs\Python\Python313

Is this a virtual environment?
True

Environment folder exists?
True

Expected environment:
D:\abhijitkk\my_env

Correct environment?
False


In [23]:
import sys
import subprocess

print("Installing LLM dependencies into:")
print(sys.executable)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-U",
    "google-generativeai",
    "python-dotenv"
])

print("\nInstallation completed successfully.")

Installing LLM dependencies into:
d:\abhijkk\my_env\Scripts\python.exe

Installation completed successfully.


In [ ]:
import google.generativeai as genai

GEMINI_API_KEY = "hh"

genai.configure(api_key=GEMINI_API_KEY)

print("Gemini API configured successfully.")

Gemini API configured successfully.


In [ ]:

# 

# complete end to end pipeline with llm

RuntimeError: Model 1 does not contain LEGITIMATE class.

In [ ]:
# ============================================================
# CYBERGUARD — COMPLETE IMPROVED DETECTION PIPELINE
# ============================================================

import re
import json
import joblib
import numpy as np
import pandas as pd

from datetime import datetime


# ============================================================
# 1. FILE PATHS
# ============================================================

MODEL_1_PATH = "cyberguard_model_1_impersonation.pkl"
TFIDF_1_PATH = "cyberguard_model_1_tfidf.pkl"

MODEL_2_PATH = "cyberguard_model_2_attack_type.pkl"
TFIDF_2_PATH = "cyberguard_model_2_tfidf.pkl"

MODEL_3_PATH = "cyberguard_model_3_behavioral.pkl"

DATASET1_PATH = "unique_professional_identities_500_repaired.csv"


# ============================================================
# 2. LOAD MODELS
# ============================================================

model_1 = joblib.load(MODEL_1_PATH)
tfidf_1 = joblib.load(TFIDF_1_PATH)

model_2 = joblib.load(MODEL_2_PATH)
tfidf_2 = joblib.load(TFIDF_2_PATH)

model_3 = joblib.load(MODEL_3_PATH)

print("=" * 70)
print("MODELS LOADED")
print("=" * 70)

print("Model 1 :", type(model_1).__name__)
print("Model 2 :", type(model_2).__name__)
print("Model 3 :", model_3.get("version", "UNKNOWN"))

print("Model 1 classes:", list(model_1.classes_))
print("Model 2 classes:", list(model_2.classes_))


# ============================================================
# 3. LOAD DATASET 1
# ============================================================

df1 = pd.read_csv(DATASET1_PATH)

required_columns = [
    "identity_id",
    "name",
    "role",
    "organization",
    "organization_type",
    "country",
    "official_email",
    "official_domain",
    "source_url",
    "source_type"
]

missing_columns = [
    col for col in required_columns
    if col not in df1.columns
]

if missing_columns:
    raise RuntimeError(
        f"Dataset 1 is missing columns: {missing_columns}"
    )

print("\nDataset 1 shape:", df1.shape)


# ============================================================
# 4. HELPER FUNCTIONS
# ============================================================

def normalize_text(value):

    value = str(value).lower().strip()

    value = re.sub(
        r"[^a-z0-9\s]",
        " ",
        value
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value


def normalize_name(value):

    value = normalize_text(value)

    titles = {
        "mr",
        "mrs",
        "ms",
        "miss",
        "dr",
        "prof",
        "sir",
        "madam"
    }

    tokens = [
        token
        for token in value.split()
        if token not in titles
    ]

    return " ".join(tokens)


def get_domain(email):

    email = str(email).lower().strip()

    if "@" not in email:
        return ""

    return email.rsplit("@", 1)[1]


def classify_risk(score):

    if score <= 20:
        return "SAFE"

    elif score <= 40:
        return "LOW"

    elif score <= 60:
        return "MEDIUM"

    elif score <= 80:
        return "HIGH"

    else:
        return "CRITICAL"


def detect_urgency(text):

    patterns = [
        r"\burgent\b",
        r"\bimmediately\b",
        r"\basap\b",
        r"\btoday\b",
        r"\bright away\b",
        r"\baction required\b",
        r"\btime sensitive\b",
        r"\bdeadline\b",
        r"\bfinal notice\b",
        r"\bwithin\s+\d+\s*(hour|hours|minute|minutes)\b",
        r"\bwill be suspended\b",
        r"\baccount will be\b"
    ]

    matches = []

    for pattern in patterns:

        if re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        ):
            matches.append(pattern)

    return len(matches) > 0, matches


def detect_sensitive_request(text):

    patterns = [
        r"\bpassword\b",
        r"\bpasscode\b",
        r"\botp\b",
        r"\bone[- ]time password\b",
        r"\bcredential(s)?\b",
        r"\blogin\b",
        r"\busername\b",
        r"\bpin\b",
        r"\baccount number\b",
        r"\bbank account\b",
        r"\bcard number\b",
        r"\bcvv\b",
        r"\bverification code\b",
        r"\bsecurity code\b",
        r"\bconfidential\b",
        r"\bsecret\b",
        r"\bsensitive data\b",
        r"\bpersonal information\b",
        r"\bidentity documents?\b"
    ]

    matches = []

    for pattern in patterns:

        if re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        ):
            matches.append(pattern)

    return len(matches) > 0, matches


def extract_urls(text):

    return re.findall(
        r"https?://[^\s<>\"]+",
        text,
        flags=re.IGNORECASE
    )


def get_url_domain(url):

    match = re.match(
        r"https?://([^/:?#]+)",
        url,
        flags=re.IGNORECASE
    )

    if match:
        return match.group(1).lower()

    return ""


def is_public_email_domain(domain):

    public_domains = {
        "gmail.com",
        "yahoo.com",
        "yahoo.co.in",
        "outlook.com",
        "hotmail.com",
        "live.com",
        "proton.me",
        "protonmail.com",
        "icloud.com",
        "aol.com",
        "mail.com"
    }

    return domain.lower() in public_domains


def domain_relationship(
    sender_domain,
    official_domain
):

    sender_domain = str(
        sender_domain
    ).lower().strip()

    official_domain = str(
        official_domain
    ).lower().strip()

    if not sender_domain or not official_domain:
        return "UNKNOWN"

    if sender_domain == official_domain:
        return "EXACT_MATCH"

    if (
        sender_domain.endswith(
            "." + official_domain
        )
        or
        official_domain.endswith(
            "." + sender_domain
        )
    ):
        return "RELATED_DOMAIN"

    return "MISMATCH"


# ============================================================
# 5. EMAIL INPUT
# ============================================================

print("\n" + "=" * 70)
print("EMAIL INPUT")
print("=" * 70)

sender_name = input(
    "Sender name: "
).strip()

sender_email = input(
    "Sender email: "
).strip().lower()

receiver_name = input(
    "Receiver name: "
).strip()

receiver_email = input(
    "Receiver email: "
).strip().lower()

reply_to = input(
    "Reply-To email (press Enter if none): "
).strip().lower()

subject = input(
    "Subject: "
).strip()

print("\nEnter message.")
print("Press ENTER on an empty line when finished.")

message_lines = []

while True:

    line = input()

    if line == "":
        break

    message_lines.append(line)

message = "\n".join(
    message_lines
).strip()

date_input = input(
    "\nDate/time (YYYY-MM-DD HH:MM, "
    "press Enter for current time): "
).strip()

if date_input:

    try:

        email_datetime = datetime.strptime(
            date_input,
            "%Y-%m-%d %H:%M"
        )

    except ValueError:

        print(
            "Invalid date format. "
            "Using current time."
        )

        email_datetime = datetime.now()

else:

    email_datetime = datetime.now()


# ============================================================
# 6. BASIC EXTRACTION
# ============================================================

sender_domain = get_domain(
    sender_email
)

receiver_domain = get_domain(
    receiver_email
)

reply_to_domain = get_domain(
    reply_to
)

text_input = (
    subject + " " + message
).strip()

text_lower = text_input.lower()


# ============================================================
# 7. DATASET 1 IDENTITY MATCHING
# ============================================================

email_matches = df1[
    df1["official_email"]
    .astype(str)
    .str.lower()
    .str.strip()
    ==
    sender_email
]

identity_found = False
identity_ambiguous = False
identity = None

if len(email_matches) == 0:

    identity_status = "UNKNOWN"

elif len(email_matches) == 1:

    identity_found = True
    identity_status = "UNIQUE"

    identity = email_matches.iloc[0]

else:

    normalized_sender_name = normalize_name(
        sender_name
    )

    name_matches = email_matches[
        email_matches["name"]
        .astype(str)
        .apply(normalize_name)
        ==
        normalized_sender_name
    ]

    if len(name_matches) == 1:

        identity_found = True

        identity_status = (
            "RESOLVED_BY_EMAIL_AND_NAME"
        )

        identity = name_matches.iloc[0]

    else:

        identity_ambiguous = True

        identity_status = "AMBIGUOUS"


# ============================================================
# 8. IDENTITY DETAILS
# ============================================================

identity_id = None
identity_name = None
identity_role = None
identity_org = None
identity_org_type = None
official_email = None
official_domain = None

print("\n" + "=" * 70)
print("IDENTITY ANALYSIS")
print("=" * 70)

print(
    "Sender:",
    sender_name
)

print(
    "Email:",
    sender_email
)

print(
    "Status:",
    identity_status
)

if identity_found:

    identity_id = str(
        identity["identity_id"]
    )

    identity_name = str(
        identity["name"]
    )

    identity_role = str(
        identity["role"]
    )

    identity_org = str(
        identity["organization"]
    )

    identity_org_type = str(
        identity["organization_type"]
    )

    official_email = str(
        identity["official_email"]
    ).lower()

    official_domain = str(
        identity["official_domain"]
    ).lower()

    print(
        "Identity ID:",
        identity_id
    )

    print(
        "Name:",
        identity_name
    )

    print(
        "Role:",
        identity_role
    )

    print(
        "Organization:",
        identity_org
    )

    print(
        "Official email:",
        official_email
    )

    print(
        "Official domain:",
        official_domain
    )

elif identity_ambiguous:

    print(
        "WARNING: Multiple identities share this "
        "official email and could not be uniquely resolved."
    )

else:

    print(
        "Sender was not found in Dataset 1."
    )


# ============================================================
# 9. MODEL 1 — IMPERSONATION DETECTION
# ============================================================

X_m1 = tfidf_1.transform(
    [text_input]
)

m1_probabilities = model_1.predict_proba(
    X_m1
)[0]

m1_classes = list(
    model_1.classes_
)

# ------------------------------------------------------------
# IMPORTANT:
# Saved Model 1 was trained using:
#
# 0 = LEGITIMATE
# 1 = IMPERSONATION
# ------------------------------------------------------------

if 0 not in m1_classes:

    raise RuntimeError(
        "Model 1 does not contain class 0 "
        "(LEGITIMATE)."
    )

if 1 not in m1_classes:

    raise RuntimeError(
        "Model 1 does not contain class 1 "
        "(IMPERSONATION)."
    )

legitimate_index = m1_classes.index(
    0
)

impersonation_index = m1_classes.index(
    1
)

legitimate_probability = float(
    m1_probabilities[
        legitimate_index
    ]
)

impersonation_probability = float(
    m1_probabilities[
        impersonation_index
    ]
)

if impersonation_probability >= 0.5:

    model1_prediction = (
        "IMPERSONATION"
    )

else:

    model1_prediction = (
        "LEGITIMATE"
    )

m1_margin = abs(
    impersonation_probability
    -
    legitimate_probability
)

if m1_margin < 0.10:

    model1_confidence = "LOW"

elif m1_margin < 0.25:

    model1_confidence = "MEDIUM"

else:

    model1_confidence = "HIGH"


print("\n" + "=" * 70)
print("MODEL 1 — IMPERSONATION DETECTION")
print("=" * 70)

print(
    "LEGITIMATE probability   :",
    f"{legitimate_probability:.4f}"
)

print(
    "IMPERSONATION probability:",
    f"{impersonation_probability:.4f}"
)

print(
    "Prediction:",
    model1_prediction
)

print(
    "Confidence:",
    model1_confidence
)


# ============================================================
# 10. MODEL 2 — ATTACK TYPE
# ============================================================

model2_prediction = (
    "NOT_APPLICABLE"
)

model2_confidence = 0.0

model2_top_predictions = []

# Model 2 must ONLY run for impersonation.
if model1_prediction == "IMPERSONATION":

    X_m2 = tfidf_2.transform(
        [text_input]
    )

    m2_probabilities = model_2.predict_proba(
        X_m2
    )[0]

    m2_classes = list(
        model_2.classes_
    )

    sorted_indices = np.argsort(
        m2_probabilities
    )[::-1]

    for index in sorted_indices[:5]:

        model2_top_predictions.append(
            {
                "attack_type": str(
                    m2_classes[index]
                ),
                "probability": float(
                    m2_probabilities[index]
                )
            }
        )

    best_index = sorted_indices[0]

    model2_prediction = str(
        m2_classes[best_index]
    )

    model2_confidence = float(
        m2_probabilities[best_index]
    )


print("\n" + "=" * 70)
print("MODEL 2 — ATTACK TYPE")
print("=" * 70)

print(
    "Prediction:",
    model2_prediction
)

if model2_prediction != "NOT_APPLICABLE":

    print(
        "Confidence:",
        f"{model2_confidence:.4f}"
    )

    print("\nTop attack types:")

    for item in model2_top_predictions:

        print(
            f"  {item['attack_type']}: "
            f"{item['probability']:.4f}"
        )

else:

    print(
        "Not applicable because Model 1 "
        "classified the email as LEGITIMATE."
    )


# ============================================================
# 11. MODEL 3 — BEHAVIORAL ANALYSIS
# ============================================================

model3_status = "NOT_AVAILABLE"

model3_normality_score = None
model3_anomaly_score = None

model3_evidence = {}

profiles = model_3.get(
    "dataset3_profiles"
)

if isinstance(
    profiles,
    pd.DataFrame
):

    profile_df = profiles.copy()

else:

    profile_df = pd.DataFrame(
        profiles
    )


if identity_found:

    profile_matches = profile_df[
        profile_df["identity_id"]
        .astype(str)
        ==
        str(identity_id)
    ]

    if len(profile_matches) == 1:

        profile = profile_matches.iloc[0]

        current_hour = (
            email_datetime.hour
        )

        current_day = (
            email_datetime.strftime("%A")
        )

        # Dataset 3 time periods
        if 5 <= current_hour < 12:

            current_period = "Morning"

        elif 12 <= current_hour < 17:

            current_period = "Afternoon"

        elif 17 <= current_hour < 22:

            current_period = "Evening"

        else:

            current_period = "Night"


        usual_hours = str(
            profile["usual_hours"]
        ).strip()

        usual_days = str(
            profile["usual_days"]
        ).strip()

        usual_sender_domain = str(
            profile["usual_sender_domain"]
        ).lower().strip()

        avg_message_length = float(
            profile["avg_message_length"]
        )

        current_message_length = len(
            message
        )


        # ----------------------------------------------------
        # TIME MATCH
        # ----------------------------------------------------

        if usual_hours == "Mixed":

            time_match = 1.0

        elif current_period == usual_hours:

            time_match = 1.0

        else:

            time_match = 0.0


        # ----------------------------------------------------
        # DAY MATCH
        # ----------------------------------------------------

        if usual_days == "All Days":

            day_match = 1.0

        elif usual_days == "Weekdays":

            if current_day in [
                "Monday",
                "Tuesday",
                "Wednesday",
                "Thursday",
                "Friday"
            ]:

                day_match = 1.0

            else:

                day_match = 0.0

        else:

            allowed_days = [
                x.strip()
                for x in usual_days.split(",")
            ]

            day_match = (
                1.0
                if current_day in allowed_days
                else 0.0
            )


        # ----------------------------------------------------
        # SENDER DOMAIN MATCH
        # ----------------------------------------------------

        sender_domain_match = (
            1.0
            if sender_domain
            ==
            usual_sender_domain
            else 0.0
        )


        # ----------------------------------------------------
        # MESSAGE LENGTH
        # ----------------------------------------------------

        if avg_message_length > 0:

            length_ratio = (
                current_message_length
                /
                avg_message_length
            )

            if (
                0.5
                <= length_ratio
                <= 1.5
            ):

                length_match = 1.0

            elif (
                0.25
                <= length_ratio
                <= 2.0
            ):

                length_match = 0.5

            else:

                length_match = 0.0

        else:

            length_ratio = None
            length_match = 0.0


        # ----------------------------------------------------
        # MODEL 3 WEIGHTS
        # ----------------------------------------------------

        weights = model_3.get(
            "weights",
            {
                "time_of_day": 0.25,
                "day_of_week": 0.20,
                "sender_domain": 0.35,
                "message_length": 0.20
            }
        )


        model3_normality_score = (

            weights["time_of_day"]
            *
            time_match

            +

            weights["day_of_week"]
            *
            day_match

            +

            weights["sender_domain"]
            *
            sender_domain_match

            +

            weights["message_length"]
            *
            length_match
        )


        model3_anomaly_score = (
            1.0
            -
            model3_normality_score
        )


        model3_status = "AVAILABLE"


        model3_evidence = {

            "current_time_period":
                current_period,

            "usual_hours":
                usual_hours,

            "time_match":
                time_match,

            "current_day":
                current_day,

            "usual_days":
                usual_days,

            "day_match":
                day_match,

            "sender_domain":
                sender_domain,

            "usual_sender_domain":
                usual_sender_domain,

            "sender_domain_match":
                sender_domain_match,

            "current_message_length":
                current_message_length,

            "average_message_length":
                avg_message_length,

            "length_ratio":
                length_ratio,

            "length_match":
                length_match
        }


print("\n" + "=" * 70)
print("MODEL 3 — BEHAVIORAL ANALYSIS")
print("=" * 70)

print(
    "Status:",
    model3_status
)

if model3_status == "AVAILABLE":

    print(
        "Normality score:",
        f"{model3_normality_score:.4f}"
    )

    print(
        "Anomaly score:",
        f"{model3_anomaly_score:.4f}"
    )

    print(
        "Time match:",
        bool(
            model3_evidence["time_match"]
        )
    )

    print(
        "Day match:",
        bool(
            model3_evidence["day_match"]
        )
    )

    print(
        "Sender domain match:",
        bool(
            model3_evidence[
                "sender_domain_match"
            ]
        )
    )

else:

    print(
        "Behavioral analysis unavailable "
        "because a unique identity profile "
        "was not established."
    )


# ============================================================
# 12. TECHNICAL ANALYSIS
# ============================================================

urls = extract_urls(
    message
)

url_information = []

for url in urls:

    domain = get_url_domain(
        url
    )

    relationship = "UNKNOWN"

    if official_domain:

        relationship = domain_relationship(
            domain,
            official_domain
        )

    url_information.append(
        {
            "url": url,
            "domain": domain,
            "relationship": relationship
        }
    )


public_sender_email = (
    is_public_email_domain(
        sender_domain
    )
)


sender_domain_identity_status = None

if identity_found:

    sender_domain_identity_status = (
        domain_relationship(
            sender_domain,
            official_domain
        )
    )


reply_to_match = None

if reply_to:

    reply_to_match = (
        reply_to
        ==
        sender_email
    )


urgency_detected, urgency_signals = (
    detect_urgency(
        text_input
    )
)


sensitive_detected, sensitive_signals = (
    detect_sensitive_request(
        text_input
    )
)


attachment_extensions = [
    ".pdf",
    ".doc",
    ".docx",
    ".xls",
    ".xlsx",
    ".csv",
    ".zip",
    ".rar",
    ".exe",
    ".js",
    ".html",
    ".htm"
]

detected_attachment_extensions = []

for extension in attachment_extensions:

    if extension in text_lower:

        detected_attachment_extensions.append(
            extension
        )

attachment_detected = (
    len(
        detected_attachment_extensions
    )
    >
    0
)


# ============================================================
# 13. RISK ENGINE
# ============================================================

risk_score = 0.0

risk_reasons = []


# ------------------------------------------------------------
# MODEL 1 — MAX 25 POINTS
# ------------------------------------------------------------

model1_points = (
    impersonation_probability
    *
    25.0
)

risk_score += model1_points


if impersonation_probability >= 0.70:

    risk_reasons.append(
        "Model 1 shows strong impersonation probability."
    )

elif impersonation_probability >= 0.50:

    risk_reasons.append(
        "Model 1 shows elevated impersonation probability."
    )


# ------------------------------------------------------------
# IDENTITY — 10 POINTS
# ------------------------------------------------------------

if identity_ambiguous:

    risk_score += 10

    risk_reasons.append(
        "Sender identity is ambiguous in Dataset 1."
    )

elif not identity_found:

    risk_score += 10

    risk_reasons.append(
        "Sender identity was not found in Dataset 1."
    )


# ------------------------------------------------------------
# DOMAIN MISMATCH — 25 POINTS
# ------------------------------------------------------------

if identity_found:

    if (
        sender_domain_identity_status
        ==
        "MISMATCH"
    ):

        risk_score += 25

        risk_reasons.append(
            "Sender domain does not match the "
            "identified professional's official domain."
        )

    elif (
        sender_domain_identity_status
        ==
        "RELATED_DOMAIN"
    ):

        risk_score += 10

        risk_reasons.append(
            "Sender domain is related to, but not "
            "exactly equal to, the official domain."
        )


# ------------------------------------------------------------
# REPLY-TO MISMATCH — 15 POINTS
# ------------------------------------------------------------

if reply_to:

    if reply_to_match is False:

        risk_score += 15

        risk_reasons.append(
            "Reply-To address differs from sender address."
        )


# ------------------------------------------------------------
# PUBLIC EMAIL — 10 POINTS
# ------------------------------------------------------------

if public_sender_email:

    risk_score += 10

    risk_reasons.append(
        "Sender uses a public email provider."
    )


# ------------------------------------------------------------
# URGENCY — 15 POINTS
# ------------------------------------------------------------

if urgency_detected:

    risk_score += 15

    risk_reasons.append(
        "Urgency or time-pressure language was detected."
    )


# ------------------------------------------------------------
# SENSITIVE REQUEST — 20 POINTS
# ------------------------------------------------------------

if sensitive_detected:

    risk_score += 20

    risk_reasons.append(
        "Sensitive information or credential-related "
        "content was detected."
    )


# ------------------------------------------------------------
# MODEL 3 — MAX 15 POINTS
# ------------------------------------------------------------

if model3_status == "AVAILABLE":

    behavior_points = (
        model3_anomaly_score
        *
        15.0
    )

    risk_score += behavior_points

    if model3_anomaly_score >= 0.50:

        risk_reasons.append(
            "Message behavior differs substantially "
            "from the identity's Dataset 3 profile."
        )


# ------------------------------------------------------------
# FINAL SCORE
# ------------------------------------------------------------

risk_score = min(
    max(
        risk_score,
        0.0
    ),
    100.0
)

risk_band = classify_risk(
    risk_score
)


# ============================================================
# 14. FINAL ATTACK TYPE
# ============================================================

if model1_prediction == "LEGITIMATE":

    final_attack_type = "NONE"

else:

    if (
        model2_prediction
        !=
        "NOT_APPLICABLE"
        and
        model2_confidence
        >=
        0.40
    ):

        final_attack_type = (
            model2_prediction
        )

    else:

        final_attack_type = (
            "IMPERSONATION_UNCERTAIN"
        )


# ============================================================
# 15. SUSPICIOUS URL EVIDENCE
# ============================================================

suspicious_urls = []

for item in url_information:

    if (
        item["relationship"]
        ==
        "MISMATCH"
    ):

        suspicious_urls.append(
            item
        )


# ============================================================
# 16. STRUCTURED EVIDENCE
# ============================================================

evidence = {

    "identity": {

        "status":
            identity_status,

        "identity_found":
            identity_found,

        "identity_ambiguous":
            identity_ambiguous,

        "identity_id":
            identity_id,

        "name":
            identity_name,

        "role":
            identity_role,

        "organization":
            identity_org,

        "organization_type":
            identity_org_type,

        "official_email":
            official_email,

        "official_domain":
            official_domain
    },


    "model_1": {

        "prediction":
            model1_prediction,

        "legitimate_probability":
            legitimate_probability,

        "impersonation_probability":
            impersonation_probability,

        "confidence":
            model1_confidence
    },


    "model_2": {

        "prediction":
            model2_prediction,

        "confidence":
            model2_confidence,

        "top_predictions":
            model2_top_predictions
    },


    "model_3": {

        "status":
            model3_status,

        "normality_score":
            model3_normality_score,

        "anomaly_score":
            model3_anomaly_score,

        "details":
            model3_evidence
    },


    "technical": {

        "sender_email":
            sender_email,

        "sender_domain":
            sender_domain,

        "receiver_email":
            receiver_email,

        "receiver_domain":
            receiver_domain,

        "reply_to":
            reply_to,

        "reply_to_match":
            reply_to_match,

        "public_sender":
            public_sender_email,

        "domain_identity_status":
            sender_domain_identity_status,

        "urls":
            url_information,

        "suspicious_urls":
            suspicious_urls,

        "attachment_detected":
            attachment_detected,

        "attachment_extensions":
            detected_attachment_extensions
    },


    "content": {

        "urgency_detected":
            urgency_detected,

        "urgency_signals":
            urgency_signals,

        "sensitive_request_detected":
            sensitive_detected,

        "sensitive_signals":
            sensitive_signals
    },


    "risk": {

        "score":
            risk_score,

        "band":
            risk_band,

        "reasons":
            risk_reasons
    },


    "final": {

        "attack_type":
            final_attack_type
    }
}


# ============================================================
# 17. GEMINI EXPLANATION LAYER
# ============================================================

GEMINI_API_KEY = (
    "PASTE_YOUR_GEMINI_API_KEY_HERE"
)

llm_status = "NOT_RUN"

llm_result = {}


if (
    GEMINI_API_KEY
    and
    GEMINI_API_KEY
    !=
    "YOUR_API_KEY_HERE"
):

    try:

        import google.generativeai as genai

        genai.configure(
            api_key=GEMINI_API_KEY
        )

        llm_model = genai.GenerativeModel(
            "gemini-3.8-flash"
        )


        prompt = f"""
You are the explanation layer of CYBERGUARD,
an AI-powered digital impersonation detection system.

The machine-learning models and deterministic risk
engine have already produced the official result.

DO NOT override, change, recalculate, or reinterpret:

Risk score: {risk_score:.2f}
Risk band: {risk_band}
Final attack type: {final_attack_type}

Use ONLY the supplied evidence.

Return VALID JSON ONLY:

{{
    "threat_summary": "...",
    "risk_level": "...",
    "primary_attack_type": "...",
    "key_evidence": [
        "...",
        "...",
        "..."
    ],
    "recommended_actions": [
        "...",
        "...",
        "..."
    ],
    "confidence_note": "..."
}}

Rules:

1. risk_level MUST be exactly "{risk_band}".
2. primary_attack_type MUST be exactly "{final_attack_type}".
3. Do not invent facts.
4. Do not claim a URL is malicious merely because it exists.
5. Do not claim an attachment is malicious merely because it exists.
6. Unknown identity does not automatically prove impersonation.
7. Model 2 is only an attack-type classifier.
8. Model 3 may be unavailable.
9. Explain evidence in simple security-analyst language.
10. Give defensive recommendations only.

EMAIL:

Sender:
{sender_name}

Sender email:
{sender_email}

Receiver:
{receiver_name}

Receiver email:
{receiver_email}

Subject:
{subject}

Message:
{message}

STRUCTURED EVIDENCE:

{json.dumps(
    evidence,
    indent=2,
    default=str
)}
"""


        response = llm_model.generate_content(
            prompt
        )

        raw_response = response.text.strip()


        # Remove Markdown JSON fences.

        raw_response = re.sub(
            r"^```json\s*",
            "",
            raw_response,
            flags=re.IGNORECASE
        )

        raw_response = re.sub(
            r"^```\s*",
            "",
            raw_response
        )

        raw_response = re.sub(
            r"\s*```$",
            "",
            raw_response
        )


        llm_result = json.loads(
            raw_response
        )


        # Force deterministic values.

        llm_result[
            "risk_level"
        ] = risk_band

        llm_result[
            "primary_attack_type"
        ] = final_attack_type

        llm_status = "SUCCESS"


    except Exception as e:

        llm_status = "ERROR"

        llm_result = {

            "threat_summary":
                "Gemini explanation failed. "
                "Use the deterministic CYBERGUARD result.",

            "risk_level":
                risk_band,

            "primary_attack_type":
                final_attack_type,

            "key_evidence":
                risk_reasons[:5],

            "recommended_actions": [

                "Verify the sender through an independent trusted channel.",

                "Do not provide passwords, OTPs, "
                "credentials, or sensitive information.",

                "Report suspicious communication "
                "to the security team."
            ],

            "confidence_note":
                str(e)
        }


else:

    llm_status = "SKIPPED"

    llm_result = {

        "threat_summary":
            "Gemini API key was not configured.",

        "risk_level":
            risk_band,

        "primary_attack_type":
            final_attack_type,

        "key_evidence":
            risk_reasons[:5],

        "recommended_actions": [

            "Verify the sender through an independent trusted channel.",

            "Do not provide passwords, OTPs, "
            "credentials, or sensitive information.",

            "Report suspicious communication "
            "to the security team."
        ],

        "confidence_note":
            "LLM layer skipped."
    }


# ============================================================
# 18. FINAL REPORT
# ============================================================

print("\n\n")

print("=" * 80)
print(
    "              CYBERGUARD FINAL THREAT REPORT"
)
print("=" * 80)


# ------------------------------------------------------------
# EMAIL
# ------------------------------------------------------------

print("\n[EMAIL]")

print(
    "Sender       :",
    sender_name
)

print(
    "Sender email :",
    sender_email
)

print(
    "Receiver     :",
    receiver_name
)

print(
    "Receiver     :",
    receiver_email
)

print(
    "Subject      :",
    subject
)

print(
    "Date         :",
    email_datetime
)


# ------------------------------------------------------------
# IDENTITY
# ------------------------------------------------------------

print("\n[IDENTITY]")

print(
    "Status       :",
    identity_status
)

if identity_found:

    print(
        "Identity ID  :",
        identity_id
    )

    print(
        "Name         :",
        identity_name
    )

    print(
        "Role         :",
        identity_role
    )

    print(
        "Organization :",
        identity_org
    )

    print(
        "Official     :",
        official_email
    )

elif identity_ambiguous:

    print(
        "Multiple identities share "
        "this email."
    )

else:

    print(
        "No matching identity found."
    )


# ------------------------------------------------------------
# MODEL 1
# ------------------------------------------------------------

print(
    "\n[MODEL 1 — IMPERSONATION]"
)

print(
    "Prediction   :",
    model1_prediction
)

print(
    "Legitimate   :",
    f"{legitimate_probability:.4f}"
)

print(
    "Impersonation:",
    f"{impersonation_probability:.4f}"
)

print(
    "Confidence   :",
    model1_confidence
)


# ------------------------------------------------------------
# MODEL 2
# ------------------------------------------------------------

print(
    "\n[MODEL 2 — ATTACK TYPE]"
)

print(
    "Prediction   :",
    model2_prediction
)

if model2_prediction != "NOT_APPLICABLE":

    print(
        "Confidence   :",
        f"{model2_confidence:.4f}"
    )


# ------------------------------------------------------------
# MODEL 3
# ------------------------------------------------------------

print(
    "\n[MODEL 3 — BEHAVIOR]"
)

print(
    "Status       :",
    model3_status
)

if model3_status == "AVAILABLE":

    print(
        "Normality    :",
        f"{model3_normality_score:.4f}"
    )

    print(
        "Anomaly      :",
        f"{model3_anomaly_score:.4f}"
    )


# ------------------------------------------------------------
# TECHNICAL
# ------------------------------------------------------------

print(
    "\n[TECHNICAL SIGNALS]"
)

print(
    "Public sender:",
    public_sender_email
)

print(
    "Reply-To     :",
    reply_to if reply_to else "None"
)

print(
    "Reply match  :",
    reply_to_match
)

print(
    "URLs detected:",
    len(urls)
)

print(
    "Attachments  :",
    attachment_detected
)

print(
    "Urgency      :",
    urgency_detected
)

print(
    "Sensitive    :",
    sensitive_detected
)


# ------------------------------------------------------------
# RISK
# ------------------------------------------------------------

print(
    "\n[RISK ENGINE]"
)

print(
    "Risk score   :",
    f"{risk_score:.2f}/100"
)

print(
    "Risk band    :",
    risk_band
)

print(
    "\nRisk reasons:"
)

if risk_reasons:

    for reason in risk_reasons:

        print(
            " -",
            reason
        )

else:

    print(
        " - No major risk signals detected."
    )


# ------------------------------------------------------------
# FINAL
# ------------------------------------------------------------

print(
    "\n[FINAL CLASSIFICATION]"
)

print(
    "Threat type  :",
    final_attack_type
)


# ------------------------------------------------------------
# GEMINI
# ------------------------------------------------------------

print(
    "\n[LLM ANALYST EXPLANATION]"
)

print(
    "LLM status   :",
    llm_status
)

print(
    "\nThreat summary:"
)

print(
    llm_result.get(
        "threat_summary",
        ""
    )
)

print(
    "\nKey evidence:"
)

for item in llm_result.get(
    "key_evidence",
    []
):

    print(
        " -",
        item
    )


print(
    "\nRecommended actions:"
)

for item in llm_result.get(
    "recommended_actions",
    []
):

    print(
        " -",
        item
    )


print(
    "\nConfidence note:"
)

print(
    llm_result.get(
        "confidence_note",
        ""
    )
)


print("\n" + "=" * 80)
print(
    "CYBERGUARD PIPELINE COMPLETED"
)
print("=" * 80)


# ============================================================
# 19. MACHINE-READABLE FINAL RESULT
# ============================================================

final_result = {

    "timestamp":
        datetime.now().isoformat(),

    "email": {

        "sender_name":
            sender_name,

        "sender_email":
            sender_email,

        "receiver_name":
            receiver_name,

        "receiver_email":
            receiver_email,

        "reply_to":
            reply_to,

        "subject":
            subject
    },

    "identity":
        evidence["identity"],

    "model_1":
        evidence["model_1"],

    "model_2":
        evidence["model_2"],

    "model_3":
        evidence["model_3"],

    "technical":
        evidence["technical"],

    "content":
        evidence["content"],

    "risk":
        evidence["risk"],

    "final":
        evidence["final"],

    "llm": {

        "status":
            llm_status,

        "result":
            llm_result
    }
}


print(
    "\nMachine-readable result stored in variable:"
)

print(
    "final_result"
)

MODELS LOADED
Model 1 : LogisticRegression
Model 2 : LogisticRegression
Model 3 : CYBERGUARD_MODEL_3_FINAL
Model 1 classes: [np.int64(0), np.int64(1)]
Model 2 classes: ['ACCOUNT_CHANGE', 'AUTHORITY_ABUSE', 'CHANNEL_CHANGE', 'CREDENTIAL_REQUEST', 'DOCUMENT_REQUEST', 'FINANCIAL_REQUEST', 'SENSITIVE_DATA_REQUEST', 'SOCIAL_ENGINEERING', 'URGENT_REQUEST', 'VENDOR_PAYMENT']

Dataset 1 shape: (500, 10)

EMAIL INPUT
